In [0]:
# Databricks notebook source

# =========================================
# 0) PARAMÈTRES
# =========================================

storage_account = "stmlopsartifacts001"
container = "ml-data"
gold_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/delta/gold/churn_training_dataset"

# Remplace par une nouvelle clé après rotation
storage_key = "HTRoBDjrf6m9Oc8kjlUA9S1qcfLg1KSjHQQb4o1NnPcMrR4gyZkpRqW8I3qtB+g5xxI9ahJ6pzYT+ASt8WM8BA=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key.strip()
)

experiment_name = "/Shared/churn-demo-training"
registered_model_name = "churn_demo_model"

# COMMAND ----------

# =========================================
# 1) IMPORTS
# =========================================

import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow.models import infer_signature

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# COMMAND ----------

# =========================================
# 2) CONFIG MLFLOW SANS UNITY CATALOG
# =========================================

mlflow.set_experiment(experiment_name)
mlflow.set_registry_uri("databricks")

# COMMAND ----------

# =========================================
# 3) CHARGER LE GOLD
# =========================================

df = spark.read.format("delta").load(gold_path)
display(df.limit(20))

pdf = df.toPandas()

if "churn_label" not in pdf.columns:
    raise ValueError("La colonne 'churn_label' est absente du dataset gold.")

pdf = pdf[pdf["churn_label"].notna()].copy()
pdf["churn_label"] = pdf["churn_label"].astype(int)

# COMMAND ----------

# =========================================
# 4) FEATURES / TARGET
# =========================================

target_col = "churn_label"

candidate_feature_cols = [
    "industry",
    "country",
    "nps_score",
    "tenure_months",
    "monthly_contract_value",
    "login_count_30d",
    "feature_a_usage_30d",
    "feature_b_usage_30d",
    "storage_used_gb",
    "api_calls_30d",
    "invoice_amount",
    "paid_on_time",
    "days_late",
    "support_tickets_90d",
    "csat_score",
    "is_high_value_customer",
    "late_payment_risk",
    "low_engagement_risk",
    "support_risk",
]

feature_cols = [c for c in candidate_feature_cols if c in pdf.columns]

if len(feature_cols) == 0:
    raise ValueError("Aucune feature disponible dans le dataset gold.")

print("Features utilisées :", feature_cols)

X = pdf[feature_cols].copy()
y = pdf[target_col].copy()

categorical_cols = [c for c in ["industry", "country"] if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# COMMAND ----------

# =========================================
# 5) PIPELINE ML
# =========================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

# COMMAND ----------

# =========================================
# 6) TRAINING + MLFLOW LEGACY REGISTRY
# =========================================

with mlflow.start_run(run_name="rf_churn_training") as run:
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    else:
        y_proba = None
        auc = None

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("feature_count", len(feature_cols))
    mlflow.log_param("feature_cols", ",".join(feature_cols))

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    if auc is not None:
        mlflow.log_metric("roc_auc", auc)

    input_example = X_test.head(5)
    signature = infer_signature(X_train, pipeline.predict(X_train))

    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature,
        registered_model_name=registered_model_name
    )

    run_id = run.info.run_id

print("run_id =", run_id)
print("model_uri =", model_info.model_uri)
print("accuracy =", acc)
print("f1_score =", f1)
print("roc_auc =", auc)